In [84]:
from pathlib import Path
from typing import List, Union
import random
import re
from datetime import timedelta
import traceback
import os
import sys
from mm_story_agent import MMStoryAgent
import yaml
import json
from tqdm import trange
import numpy as np
import librosa
import cv2
import shutil
import traceback
# Import moviepy config first
import moviepy.config as mpconfig
import pandas as pd 
import numpy as np 

In [9]:
import moviepy.editor as mpy
from moviepy.audio.AudioClip import AudioArrayClip
from moviepy.audio.fx.all import audio_loop
from moviepy.video.fx.all import resize
from moviepy.video.tools.subtitles import SubtitlesClip

from mm_story_agent.base import register_tool

# Update the class references to use mpy instead
ImageClip = mpy.ImageClip
AudioFileClip = mpy.AudioFileClip
CompositeAudioClip = mpy.CompositeAudioClip
CompositeVideoClip = mpy.CompositeVideoClip
ColorClip = mpy.ColorClip
VideoFileClip = mpy.VideoFileClip
VideoClip = mpy.VideoClip
TextClip = mpy.TextClip
concatenate_audioclips = mpy.concatenate_audioclips
concatenate_videoclips = mpy.concatenate_videoclips


Initializing ToolRegistry
Importing all tools...
Registering QAOutlineStoryWriter...
Registering qa_outline_story_writer: QAOutlineStoryWriter
Registering tool: qa_outline_story_writer
Registering LLM agents...
Registering gemini: GeminiAgent
Registering tool: gemini
Registering openai: OpenAIAgent
Registering tool: openai
Registering musicgen_t2m: MusicGenAgent
Registering tool: musicgen_t2m
Registering audioldm2_t2a: AudioLDM2Agent
Registering tool: audioldm2_t2a
Registering gtts: GoogleTTSAgent
Registering tool: gtts
Registering elevenlabs: ElevenLabsAgent
Registering tool: elevenlabs
Registering story_diffusion_t2i: StoryDiffusionAgent
Registering tool: story_diffusion_t2i
Registering freesound_sfx_retrieval: FreesoundSfxAgent
Registering tool: freesound_sfx_retrieval
Registering freesound_music_retrieval: FreesoundMusicAgent
Registering tool: freesound_music_retrieval
Using ImageMagick from: C:\Program Files\ImageMagick-7.1.1-Q16-HDRI\magick.exe
Registering slideshow_video_compose

In [10]:
def ensure_imagemagick():
    """Ensure ImageMagick is properly configured"""
    if os.name == 'nt':  # Windows
        # Try to find ImageMagick installation
        possible_paths = [
            r"C:\Program Files\ImageMagick-7.1.1-Q16-HDRI\magick.exe",
            r"C:\Program Files\ImageMagick-7.1.1-Q16\magick.exe",
            r"C:\Program Files (x86)\ImageMagick-7.1.1-Q16-HDRI\magick.exe",
            r"C:\Program Files (x86)\ImageMagick-7.1.1-Q16\magick.exe",
        ]
        
        imagemagick_path = None
        for path in possible_paths:
            if os.path.exists(path):
                imagemagick_path = path
                break
        
        if imagemagick_path:
            mpconfig.change_settings({"IMAGEMAGICK_BINARY": imagemagick_path})
            print(f"Using ImageMagick from: {imagemagick_path}")
        else:
            print("Warning: ImageMagick not found in common locations")
            mpconfig.change_settings({"IMAGEMAGICK_BINARY": "magick"})

In [11]:
def add_bottom_black_area(clip: VideoFileClip,
                          black_area_height: int = 64):
    """
    Add a black area at the bottom of the video clip (for captions).

    Args:
        clip (VideoFileClip): Video clip to be processed.
        black_area_height (int): Height of the black area.

    Returns:
        VideoFileClip: Processed video clip.
    """
    black_bar = ColorClip(size=(clip.w, black_area_height), color=(255, 255, 255), duration=clip.duration)
    extended_clip = CompositeVideoClip([clip, black_bar.set_position(("center", "bottom"))])
    return extended_clip

In [12]:
def add_zoom_effect(clip, speed=1.0, mode='in', position='center'):
    fps = clip.fps
    duration = clip.duration
    total_frames = int(duration * fps)
    def main(getframe, t):
        frame = getframe(t)
        h, w = frame.shape[: 2]
        i = t * fps
        if mode == 'out':
            i = total_frames - i
        zoom = 1 + (i * ((0.1 * speed) / total_frames))
        positions = {'center':  [(w - (w * zoom)) / 2,  (h - (h  *  zoom)) / 2],
                     'left': [0, (h - (h * zoom)) / 2],
                     'right': [(w - (w * zoom)), (h - (h * zoom)) / 2],
                     'top': [(w - (w * zoom)) / 2, 0],
                     'topleft': [0, 0],
                     'topright': [(w - (w * zoom)), 0],
                     'bottom': [(w - (w * zoom)) / 2, (h - (h * zoom))],
                     'bottomleft': [0, (h - (h * zoom))],
                     'bottomright': [(w - (w * zoom)), (h - (h * zoom))]}
        tx, ty = positions[position]
        M = np.array([[zoom, 0, tx], [0, zoom, ty]])
        frame = cv2.warpAffine(frame, M, (w, h))
        return frame
    return clip.fl(main)

In [45]:
def add_move_effect(clip, direction="left", move_raito=0.9):
    """Add pan left/right effect with dynamic zoom to video clip"""
    w, h = clip.size
    move_distance = int(w * (1 - move_raito))
    
    # Increase zoom factor significantly to ensure no black edges
    
    # required_zoom = 1.3  # Fixed larger zoom
    zoom = (w + move_distance) / w
    def effect(get_frame, t):
        progress = t / clip.duration
        frame = get_frame(t)
        zx = zoom 
        zy = zoom
        tx = (-zoom +1 )*w/2
        ty = (-zoom +1 )*h/2
        z = np.array([[zx, 0, tx], [0, zy, ty]], dtype=np.float32)
        zoomed = cv2.warpAffine(frame, z, (w, h), flags= cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)

        if direction.lower() == "left":
            dx = -move_distance*progress
        else:
            dx = move_distance*(progress)
        M = np.array([[1, 0, dx], [0, 1, 0]], dtype=np.float32)
        moved = cv2.warpAffine(zoomed, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
        return moved
    return clip.fl(effect)
       

In [ ]:
list1 = [1,2,3,4,5]
list2 = [6,7,8,9,10]
list3 = [11,12,13,14,15]
for ls, ls2, ls3 in zip(list1, list2, list3):
    print(ls, ls2, ls3)


ValueError: not enough values to unpack (expected 4, got 3)

In [150]:
def combine_chars_to_words(csv_path):
    df = pd.read_csv(csv_path)
    chars = df['a'].tolist()
    chars = chars[1:]
    starts = df['b'].tolist()
    starts = starts[1:]
    ends = df['c'].tolist()
    ends = ends[1:]
    words = []
    words_starts = []
    words_ends = []
    word = ""
    start_list = []
    end_list = []
    n = len(chars)
    for i in  range(n):
        char = chars[i]
        start = starts[i]
        end = ends[i]
        
        if char == " ":
            words.append(word)
            start_time = start_list[0]
            end_time = end_list[-1]
            start_list = []
            end_list = []
            words_starts.append(start_time)
            words_ends.append(end_time)
            word = ""
        else:
            word +=char
            start_list.append(start)
            end_list.append(end)
        
        if i ==n-1:
            words.append(word)
            start_time = start_list[0]
            end_time = end_list[-1]
            
            words_starts.append(start_time)
            words_ends.append(end_time)
            word = ""
            start_list = []
            end_list = []
    

    df = pd.DataFrame({"words": words, "words_starts": words_starts, "words_ends": words_ends})
    new_csv_path = csv_path.replace(".csv", "_combined.csv")
    df.to_csv(new_csv_path, index=False)
    return words, words_starts, words_ends

In [135]:
csv = "generated_stories\example\speech\p1.csv"
combine_chars_to_words(csv)

(['Ever',
  'seen',
  'two',
  'best',
  'friends',
  'lose',
  'everything',
  'over',
  'a',
  'silly',
  'fight?',
  'Let’s',
  'jump',
  'in!'],
 [0.0,
  0.255,
  0.453,
  0.639,
  0.859,
  1.161,
  1.405,
  1.881,
  2.136,
  2.206,
  2.485,
  3.146,
  3.379,
  3.611],
 [0.209,
  0.418,
  0.569,
  0.813,
  1.103,
  1.347,
  1.823,
  2.078,
  2.159,
  2.426,
  2.868,
  3.344,
  3.553,
  4.04])

In [180]:
def words_to_combined_csv(csv_list, slide_duration=0.1, fade_duration = 0.1, fps = 24):
    combined_words = []
    combined_words_starts = []
    combined_words_ends = []
    previous_end = 0
    total_duration_prev = 0
    total_duration = 0
    for i in range(len(csv_list)):
        words, words_starts, words_ends = combine_chars_to_words(csv_list[i])
        combined_words.extend(words)
        corrected_words_starts = [start +previous_end +2*(slide_duration +fade_duration) for start in words_starts]
        corrected_words_ends = [end +previous_end +2*(slide_duration +fade_duration) for end in words_ends]
        combined_words_starts.extend(corrected_words_starts)
        combined_words_ends.extend(corrected_words_ends)
        previous_end = corrected_words_ends[-1]
        total_duration_prev += words_ends[-1]
        total_duration += words_ends[-1] + 2*(slide_duration +fade_duration)

    return combined_words, combined_words_starts, combined_words_ends, total_duration, total_duration_prev





In [181]:
csv_list = []
csv_list_path = "generated_stories\example\speech"
num_files = 0
for file in os.listdir(csv_list_path):
    if file.endswith(".csv"):
        num_files +=1
num_files = num_files//2
for i in range(num_files):
    csv_list.append(csv_list_path + f"\p{i+1}.csv")

words, words_starts, words_ends, total_duration, total_duration_prev = words_to_combined_csv(csv_list)
print(words[:25])
print(words_starts[:25])
print(words_ends[:25])
print(total_duration)
print(total_duration_prev)




['Ever', 'seen', 'two', 'best', 'friends', 'lose', 'everything', 'over', 'a', 'silly', 'fight?', 'Let’s', 'jump', 'in!', 'In', 'a', 'quiet', 'village,', 'two', 'cats', 'were', 'the', 'closest', 'pals—until', 'one']
[0.4, 0.655, 0.853, 1.0390000000000001, 1.259, 1.561, 1.8050000000000002, 2.281, 2.536, 2.606, 2.885, 3.546, 3.779, 4.011, 4.840000000000001, 5.014000000000001, 5.119000000000001, 5.479000000000001, 6.140000000000001, 6.454000000000001, 6.790000000000001, 6.976000000000001, 7.0920000000000005, 7.557, 8.869000000000002]
[0.609, 0.8180000000000001, 0.969, 1.213, 1.5030000000000001, 1.7469999999999999, 2.223, 2.4779999999999998, 2.5589999999999997, 2.826, 3.268, 3.7439999999999998, 3.953, 4.44, 4.968000000000001, 5.037000000000001, 5.409000000000001, 5.955000000000001, 6.384, 6.744000000000001, 6.941000000000001, 7.046000000000001, 7.4990000000000006, 8.787, 9.124]
79.72800000000001
73.328


In [182]:
dataframe = pd.DataFrame({"words": words, "words_starts": words_starts, "words_ends": words_ends})
print(dataframe)


       words  words_starts  words_ends
0       Ever         0.400       0.609
1       seen         0.655       0.818
2        two         0.853       0.969
3       best         1.039       1.213
4    friends         1.259       1.503
..       ...           ...         ...
197   before        77.894      78.161
198     your        78.196      78.323
199   snacks        78.370      78.718
200   vanish        78.776      79.101
201     too!        79.159      79.728

[202 rows x 3 columns]


: 

In [46]:
def add_slide_effect(clips, slide_duration=0.4):
    """Concatenate clips with sliding transition"""
    final_clips = []
    for i, clip in enumerate(clips):
        if i > 0:  # Add transition for all clips except first
            clip = clip.crossfadein(slide_duration)
        final_clips.append(clip)
    return concatenate_videoclips(final_clips, method="compose")

In [59]:
def morph_transition(img1_path, img2_path, duration=0.5, fps=10):
    """
    Morph one image into another using weighted blending.
    Returns a list of ImageClips.
    """
    img1 = cv2.imread(str(img1_path))
    img2 = cv2.imread(str(img2_path))

    # Resize second image to match first
    img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))

    n_frames = int(0.5 * fps)
    clips = []

    for i in range(n_frames):
        alpha = (i + 1) / n_frames
        beta = 1.0 - alpha
        blended = cv2.addWeighted(img1, beta, img2, alpha, 0.0)
        blended_rgb = cv2.cvtColor(blended, cv2.COLOR_BGR2RGB)
        clip = ImageClip(blended_rgb).set_duration(1/fps)
        clips.append(clip)

    return clips



In [60]:
def morph_transition(img1_path, img2_path, duration=0.5, fps=24, zoom=True, glitch=False, blur=True):
    img1 = cv2.imread(str(img1_path))
    img2 = cv2.imread(str(img2_path))
    img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))

    n_frames = int(duration * fps)
    clips = []

    for i in range(n_frames):
        alpha = (i + 1) / n_frames
        beta = 1.0 - alpha

        blended = cv2.addWeighted(img1, beta, img2, alpha, 0.0)

        if blur:
            blur_strength = int(11 * (1 - abs(0.5 - alpha) * 2))
            blended = cv2.GaussianBlur(blended, (blur_strength|1, blur_strength|1), 0)

        if glitch:
            shift = int(5 * np.sin(alpha * np.pi))
            blended[:, :, 0] = np.roll(blended[:, :, 0], shift, axis=1)

        if zoom:
            scale = 1.1 - 0.1 * abs(0.5 - alpha) * 2
            h, w = blended.shape[:2]
            center_x, center_y = w // 2, h // 2
            new_w, new_h = int(w * scale), int(h * scale)
            resized = cv2.resize(blended, (new_w, new_h))
            # Center crop
            start_x = (new_w - w) // 2
            start_y = (new_h - h) // 2
            blended = resized[start_y:start_y + h, start_x:start_x + w]

        rgb = cv2.cvtColor(blended, cv2.COLOR_BGR2RGB)
        clips.append(ImageClip(rgb).set_duration(1 / fps))

    return clips


In [61]:

def add_morph_transition(clips, image_dir, duration=0.5, fps=10):
    final = []
    for i in range(len(clips)):
        final.append(clips[i])
        if i < len(clips) - 1:
            img1_path = image_dir / f"p{i+1}.png"
            img2_path = image_dir / f"p{i+2}.png"
            morph_clips = morph_transition(img1_path, img2_path, 0.5, fps)
            final.extend(morph_clips)
    return concatenate_videoclips(final, method="compose")

In [77]:

def slide_push_transition(img1_path: Path, img2_path: Path, duration: float = 0.5, fps: int = 24):
    """
    Push-transition: Image 1 slides out to left as Image 2 slides in from right.
    """
    img1 = cv2.imread(str(img1_path))
    img2 = cv2.imread(str(img2_path))
    h, w = img1.shape[:2]
    img2 = cv2.resize(img2, (w, h))

    n_frames = int(duration * fps)
    clips = []

    for i in range(n_frames):
        progress = i / (n_frames - 1)
        dx = int(w * progress)
        frame = np.zeros_like(img1)
        # portion of img1 sliding out
        frame[:, :w-dx] = img1[:, dx:]
        # portion of img2 sliding in
        frame[:, w-dx:] = img2[:, :dx]

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        clips.append(ImageClip(rgb).set_duration(1/fps))

    return clips

def glitch_transition(img1_path: Path, img2_path: Path, duration: float = 0.5, fps: int = 24):
    """
    Glitch transition: Blend with RGB channel shifts for a digital glitch effect.
    """
    img1 = cv2.imread(str(img1_path))
    img2 = cv2.imread(str(img2_path))
    h, w = img1.shape[:2]
    img2 = cv2.resize(img2, (w, h))

    n_frames = int(duration * fps)
    clips = []

    for i in range(n_frames):
        alpha = (i + 1) / n_frames
        blended = cv2.addWeighted(img1, 1 - alpha, img2, alpha, 0.0)

        # shift red channel sinusoidally
        r, g, b = cv2.split(blended)
        shift = int(5 * np.sin(alpha * np.pi))
        r = np.roll(r, shift, axis=1)
        glitch = cv2.merge((r, g, b))

        rgb = cv2.cvtColor(glitch, cv2.COLOR_BGR2RGB)
        clips.append(ImageClip(rgb).set_duration(1/fps))

    return clips

def blur_zoom_transition(img1_path: Path, img2_path: Path, duration: float = 0.5, fps: int = 24):
    """
    Blur+Zoom transition: Mid-blur for dream effect + slight zoom zooms frames.
    """
    img1 = cv2.imread(str(img1_path))
    img2 = cv2.imread(str(img2_path))
    h, w = img1.shape[:2]
    img2 = cv2.resize(img2, (w, h))

    n_frames = int(duration * fps)
    clips = []

    for i in range(n_frames):
        alpha = (i + 1) / n_frames
        blended = cv2.addWeighted(img1, 1 - alpha, img2, alpha, 0.0)

        # dynamic blur strength
        blur_strength = int(11 * (1 - abs(0.5 - alpha) * 2)) | 1
        blurred = cv2.GaussianBlur(blended, (blur_strength, blur_strength), 0)

        # zoom effect centered
        scale = 1 + 0.1 * (alpha - 0.5)
        new_w, new_h = int(w * scale), int(h * scale)
        resized = cv2.resize(blurred, (new_w, new_h))
        start_x, start_y = (new_w - w) // 2, (new_h - h) // 2
        cropped = resized[start_y:start_y+h, start_x:start_x+w]

        rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
        clips.append(ImageClip(rgb).set_duration(1/fps))

    return clips

# Example integration wrapper
def add_dynamic_transitions(video_clips, image_dir: Path, fps: int = 24, duration: float = 0.5):
    """
    Injects dynamic transitions between each pair of video_clips using the 3 effects in rotation.
    """
    transition_funcs = [slide_push_transition, glitch_transition, blur_zoom_transition]
    output_clips = []

    for idx, clip in enumerate(video_clips):
        output_clips.append(clip)
        if idx < len(video_clips) - 1:
            img1 = image_dir / f"p{idx+1}.png"
            img2 = image_dir / f"p{idx+2}.png"
            fn = transition_funcs[idx % len(transition_funcs)]
            trans_clips = fn(img1, img2, duration=0.5, fps=fps)
            output_clips.extend(trans_clips)
            output = concatenate_videoclips(output_clips, method="compose")

    return output

In [81]:
def compose_video(story_dir: Union[str, Path],
                  save_path: Union[str, Path],
                  captions: List,
                  music_path: Union[str, Path],
                  num_pages: int,
                  fps: int = 24,
                  audio_sample_rate: int = 16000,
                  audio_codec: str = "mp3",
                  caption_config: dict = {},
                  fade_duration: float = 0.1,
                  slide_duration: float = 0.1,
                  zoom_speed: float = 0.05,
                  move_ratio: float = 0.95,
                  sound_volume: float = 0.0,
                  music_volume: float = 0.2,
                  bg_speech_ratio: float = 0.4):
    try:
        if not isinstance(story_dir, Path):
            story_dir = Path(story_dir)

        sound_dir = story_dir / "sound"
        image_dir = story_dir / "image"
        speech_dir = story_dir / "speech"
        slide_duration = 0.5

        print("\nProcessing video with parameters:")
        print(f"Story directory: {story_dir}")
        print(f"Save path: {save_path}")
        print(f"Number of pages: {num_pages}")
        print(f"FPS: {fps}")
        print(f"Audio sample rate: {audio_sample_rate}")

    
        video_clips = []
        cur_duration = 0
        timestamps = []

        for page in trange(1, num_pages + 1):
            print(f"\nProcessing page {page}")
            
            # Create silence clips
            slide_silence = AudioArrayClip(np.zeros((int(audio_sample_rate * slide_duration), 2)), fps=audio_sample_rate)
            fade_silence = AudioArrayClip(np.zeros((int(audio_sample_rate * fade_duration), 2)), fps=audio_sample_rate)

            # Process speech
            speech_file = speech_dir / f"p{page}.wav"
            if not speech_file.exists():
                print(f"Warning: Speech file not found for page {page}")
                continue

            speech_clip = AudioFileClip(str(speech_file), fps=audio_sample_rate)
            speech_clip = concatenate_audioclips([fade_silence, speech_clip, fade_silence])
            
            # Add slide silence
            if page == 1:
                speech_clip = concatenate_audioclips([speech_clip, slide_silence])
            else:
                speech_clip = concatenate_audioclips([slide_silence, speech_clip, slide_silence])

            # Add timestamp
            timestamps.append([cur_duration + fade_duration,
                             cur_duration + speech_clip.duration - fade_duration - slide_duration])
            cur_duration += speech_clip.duration - slide_duration

            # Process image
            image_file = image_dir / f"p{page}.png"
            if not image_file.exists():
                print(f"Warning: Image file not found for page {page}")
                continue

            image_clip = ImageClip(str(image_file))
            image_clip = image_clip.set_duration(speech_clip.duration).set_fps(fps)
            image_clip = image_clip.crossfadein(fade_duration).crossfadeout(fade_duration)

            # Add effects
            if random.random() <= 0.5:
                zoom_mode = "in" if random.random() <= 0.5 else "out"
                image_clip = add_zoom_effect(image_clip, zoom_speed, zoom_mode)
            else:
                direction = "left" if random.random() <= 0.5 else "right"
                image_clip = add_move_effect(image_clip, direction=direction, move_raito=move_ratio)

            # Process sound
            sound_file = sound_dir / f"p{page}.wav"
            audio_clip = speech_clip  # Default to speech only
            if sound_file.exists():
                try:
                    sound_clip = AudioFileClip(str(sound_file), fps=audio_sample_rate)
                    sound_clip = sound_clip.audio_fadein(fade_duration)
                    if sound_clip.duration < speech_clip.duration:
                        sound_clip = audio_loop(sound_clip, duration=speech_clip.duration)
                    else:
                        sound_clip = sound_clip.subclip(0, speech_clip.duration)
                    audio_clip = CompositeAudioClip([speech_clip, sound_clip.volumex(sound_volume)])
                except Exception as e:
                    print(f"Error processing sound for page {page}: {str(e)}")

            video_clip = image_clip.set_audio(audio_clip)
            video_clips.append(video_clip)
            print(f"Successfully created video clip for page {page}")

        if not video_clips:
            print("Error: No video clips were created")
            return None

        print(f"\nSuccessfully created {len(video_clips)} video clips")
        print("\nComposing final video...")

        # composite_clip = add_slide_effect(video_clips, slide_duration=slide_duration)
        # composite_clip = add_morph_transition(video_clips, image_dir=image_dir, duration=slide_duration, fps=fps)
        composite_clip = add_dynamic_transitions(video_clips, story_dir/"image", fps=24, duration=0.5)
        composite_clip = add_bottom_black_area(composite_clip, black_area_height=caption_config.get("area_height", 100))
        composite_clip1 = composite_clip
        composite_clip2 = composite_clip
        # Create a copy of caption_config to avoid modifying the original
        caption_config_copy = caption_config.copy()
        max_caption_length = caption_config_copy.pop("max_length", 100)
        area_height = caption_config_copy.pop("area_height", 100)
        
        # Ensure required caption parameters are set
        caption_config_copy.update({
            "font": caption_config_copy.get("font", "Arial"),
            "fontsize": caption_config_copy.get("fontsize", 12),
            "color": caption_config_copy.get("color", "white"),
            "stroke_color": caption_config_copy.get("stroke_color", "black"),
            "stroke_width": caption_config_copy.get("stroke_width", 2),
            "area_height": area_height
        })
        
        # caption is not needed as of now as we will build it later
        # composite_clip = add_caption(
        #     captions,
        #     story_dir / "captions.srt",
        #     timestamps,
        #     composite_clip,
        #     max_caption_length,
        #     **caption_config_copy
        # )

        
        # Add music if available

        if music_path and Path(music_path).exists():
            music_clip = AudioFileClip(str(music_path), fps=audio_sample_rate)
            if music_clip.duration < composite_clip.duration:
                music_clip = audio_loop(music_clip, duration=composite_clip.duration)
            else:
                music_clip = music_clip.subclip(0, composite_clip.duration)
            all_audio_clip = CompositeAudioClip([composite_clip.audio, music_clip.volumex(music_volume)])
            composite_clip = composite_clip.set_audio(all_audio_clip)
            composite_clip1 = composite_clip1.set_audio(all_audio_clip)
        
        save_path1 = Path(str(save_path).replace(".mp4", "_without_subtitles.mp4"))
        save_path2 = Path(str(save_path).replace(".mp4", "_without_music_subtitles.mp4"))
        print(f"\nWriting video to {save_path2}")
        print(f"\nWriting video to {save_path1}")
        print(f"\nWriting video to {save_path}")
        try:
            composite_clip1.write_videofile(
                str(save_path1),
                fps=fps,
                codec='libx264',
                audio_codec=audio_codec,
                audio_fps=audio_sample_rate,
                preset='ultrafast',
                threads=4,
                ffmpeg_params=["-pix_fmt", "yuv420p"]
            )
            # composite_clip2.write_videofile(
            #     str(save_path2),
            #     fps=fps,
            #     codec='libx264',
            #     audio_codec=audio_codec,
            #     audio_fps=audio_sample_rate,
            #     preset='ultrafast',
            #     threads=4,
            #     ffmpeg_params=["-pix_fmt", "yuv420p"]
            # )
            # composite_clip.write_videofile(
            #     str(save_path),
            #     fps=fps,
            #     codec='libx264',
            #     audio_codec=audio_codec,
            #     audio_fps=audio_sample_rate,
            #     preset='ultrafast',
            #     threads=4,
            #     ffmpeg_params=["-pix_fmt", "yuv420p"]
            # )
            if Path(save_path).exists() and Path(save_path1).exists():
                print(f"Video file successfully written to {save_path}")
                print(f"Video file successfully written to {save_path1}")
                return composite_clip
            else:
                print(f"Error: Video file was not created at {save_path}")
                return None
            
                
        except Exception as e:
            print(f"Error writing video file: {str(e)}")
            traceback.print_exc()
            return None

        
     
        
        

    except Exception as e:
        print(f"Error in video composition: {str(e)}")
        traceback.print_exc()
        return None


In [166]:
@register_tool("slideshow_video_compose")
class SlideshowVideoComposeAgent:
    def __init__(self, cfg) -> None:
        self.cfg = cfg
        
    def call(self, params):
        try:
            story_dir = Path(params["story_dir"])
            save_path = params["save_path"]
            captions = params["captions"]
            music_path = params["music_path"]
            num_pages = params["num_pages"]
            
            print(f"\nStarting video composition with:")
            print(f"Story directory: {story_dir}")
            print(f"Save path: {save_path}")
            print(f"Number of pages: {num_pages}")
            print(f"Captions: {captions}")
            
            # Get configuration parameters with defaults
            config = {
                "fps": self.cfg.get("fps", 10),
                "audio_sample_rate": self.cfg.get("audio_sample_rate", 16000),
                "audio_codec": self.cfg.get("audio_codec", "mp3"),
                "fade_duration": self.cfg.get("fade_duration", 0.1),
                "slide_duration": self.cfg.get("slide_duration", 0.1),
                "zoom_speed": self.cfg.get("zoom_speed", 0.5),
                "move_ratio": self.cfg.get("move_ratio", 0.95),
                "sound_volume": self.cfg.get("sound_volume", 0.2),
                "music_volume": self.cfg.get("music_volume", 0.2),
                "bg_speech_ratio": self.cfg.get("bg_speech_ratio", 0.4),
                "caption_config": {
                    "font": self.cfg.get("caption", {}).get("font", "Arial"),
                    "fontsize": self.cfg.get("caption", {}).get("fontsize", 24),  # Increased font size
                    "color": "white",  # Force white color
                    "stroke_color": "black",  # Force black stroke
                    "stroke_width": 2,  # Increased stroke width for better visibility
                    "area_height": self.cfg.get("caption", {}).get("area_height", 120),
                    "max_length": self.cfg.get("caption", {}).get("max_length", 100)
                }
            }
            
            print(f"Caption configuration: {config['caption_config']}")
            
            # Create temporary directory for intermediate files
            temp_dir = Path(save_path).parent / "temp"
            temp_dir.mkdir(exist_ok=True, parents=True)
            
            try:
                result = compose_video(
                    story_dir=story_dir,
                    save_path=save_path,
                    captions=captions,
                    music_path=music_path,
                    num_pages=num_pages,
                    **config
                )
                
                if result is not None:
                    print(f"Video successfully saved to: {save_path}")
                    return result
                else:
                    print("Error: Video composition failed")
                    return None
                    
            finally:
                # Clean up temporary directory
                try:
                    import shutil
                    shutil.rmtree(temp_dir)
                except Exception as e:
                    print(f"Warning: Could not clean up temp directory: {e}")
                
        except Exception as e:
            print(f"Error in video composition: {str(e)}")
            import traceback
            traceback.print_exc()
            return None

Registering slideshow_video_compose: SlideshowVideoComposeAgent
Registering tool: slideshow_video_compose


In [168]:
try:
    # Load the config
    with open('configs/mm_story_agent.yaml', 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)

    # Initialize the agent
    agent = MMStoryAgent()

    # Source directory where assets are stored
    source_dir = Path("test")
    print(f"Source directory: {source_dir.absolute()}")

    # Try to load the original story text
    story_text = []
    try:
        # Try to load from script_data.json first
        if (source_dir / "script_data.json").exists():
            with open(source_dir / "script_data.json", "r") as f:
                script_data = json.load(f)
                story_text = [page["story"] for page in script_data["pages"]]
                print("Loaded story text from script_data.json")
        else:
            # Try to load from text files
            text_dir = source_dir / "text"
            if text_dir.exists():
                text_files = sorted(text_dir.glob("p*.txt"))
                for txt_file in text_files:
                    with open(txt_file, "r", encoding='utf-8') as f:
                        story_text.append(f.read().strip())
                print("Loaded story text from text files")
    except Exception as e:
        print(f"Error loading story text: {e}")
        traceback.print_exc()

    if not story_text:
        print("Warning: Could not load original story text!")

    # not needed now as currently we are using for testing only so copying is not needed

    # Create a new directory for the final video
    # output_dir = Path("test/final_output")
    # output_dir.mkdir(exist_ok=True, parents=True)
    # print(f"Output directory created at: {output_dir.absolute()}")
    # # Copy required assets to new directory
    # print("\nCopying assets to new directory...")
    # for subdir in ['image', 'sound', 'speech', 'music']:
    #     src_dir = source_dir / subdir
    #     dst_dir = output_dir / subdir
    #     if src_dir.exists():
    #         print(f"Copying {subdir} files...")
    #         shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)

    # Update config to use new directory
    config["story_dir"] = str(source_dir)

    # Try to get pages from image files
    image_dir = source_dir / "image"
    if image_dir.exists():
        image_files = sorted(list(image_dir.glob('p*.png')))
        if image_files:
            print(f"\nFound {len(image_files)} image files")
            
            # Use original story text if available, otherwise use generic pages
            if story_text and len(story_text) == len(image_files):
                pages = story_text
                print("Using original story text for captions")
            else:
                pages = [f"Page {i+1}" for i in range(len(image_files))]
                print("Using generic page numbers for captions")
            
            # Create script data
            script_data = {"pages": [{"story": page} for page in pages]}
            
            print("\nChecking required files:")
            for idx in range(len(pages)):
                print(f"\nPage {idx+1}:")
                print(f"Image exists: {(image_dir / f'p{idx+1}.png').exists()}")
                print(f"Speech exists: {(source_dir / 'speech' / f'p{idx+1}.wav').exists()}")
                print(f"Sound exists: {(source_dir / 'sound' / f'p{idx+1}.wav').exists()}")
            print(f"Music exists: {(source_dir / 'music' / 'music.wav').exists()}")
            
            try:
                print("\nStarting video composition...")
                result = agent.compose_video(config, pages, script_data)
                
                if result is not None:
                    output_video = source_dir / "final_video.mp4"
                    if output_video.exists():
                        print(f"\nSuccess! Video saved at: {output_video.absolute()}")
                        # Try to copy back to original directory
                        try:
                            shutil.copy2(output_video, source_dir / "final_video.mp4")
                            print(f"Video also copied to: {source_dir / 'final_video.mp4'}")
                        except Exception as e:
                            print(f"Could not copy video back to source directory: {e}")
                    else:
                        print("\nError: Video file not found after composition!")
                else:
                    print("\nError: Video composition returned None!")
                    
            except Exception as e:
                print(f"\nError during video composition: {str(e)}")
                traceback.print_exc()
        else:
            print("No image files found with pattern p*.png")
    else:
        print(f"Image directory not found at {image_dir}")

except Exception as e:
    print(f"Fatal error: {str(e)}")
    traceback.print_exc()
    sys.exit(1) 

Traceback (most recent call last):
  File "C:\Users\rajad\AppData\Local\Temp\ipykernel_9820\699382588.py", line 19, in <module>
    script_data = json.load(f)
  File "c:\Users\rajad\anaconda3\envs\story\lib\json\__init__.py", line 293, in load
    return loads(fp.read(),
  File "c:\Users\rajad\anaconda3\envs\story\lib\codecs.py", line 322, in decode
    (result, consumed) = self._buffer_decode(data, self.errors, final)
UnicodeDecodeError: 'utf-8' codec can't decode byte 0x97 in position 308: invalid start byte


Source directory: c:\Users\rajad\AptSmart\story\MM_StoryAgent\test
Error loading story text: 'utf-8' codec can't decode byte 0x97 in position 308: invalid start byte

Found 18 image files
Using generic page numbers for captions

Checking required files:

Page 1:
Image exists: True
Speech exists: True
Sound exists: False

Page 2:
Image exists: True
Speech exists: True
Sound exists: False

Page 3:
Image exists: True
Speech exists: True
Sound exists: False

Page 4:
Image exists: True
Speech exists: True
Sound exists: False

Page 5:
Image exists: True
Speech exists: True
Sound exists: False

Page 6:
Image exists: True
Speech exists: True
Sound exists: False

Page 7:
Image exists: True
Speech exists: True
Sound exists: False

Page 8:
Image exists: True
Speech exists: True
Sound exists: True

Page 9:
Image exists: True
Speech exists: True
Sound exists: False

Page 10:
Image exists: True
Speech exists: True
Sound exists: False

Page 11:
Image exists: True
Speech exists: True
Sound exists: Fal

  0%|          | 0/18 [00:00<?, ?it/s]


Processing page 1


  6%|▌         | 1/18 [00:00<00:04,  4.00it/s]

...

100%|██████████| 18/18 [00:03<00:00,  4.64it/s]

Successfully created video clip for page 17

Processing page 18
Successfully created video clip for page 18

Successfully created 18 video clips

Composing final video...



Writing video to test\final_video_without_music_subtitles.mp4

Writing video to test\final_video_without_subtitles.mp4

Writing video to test\final_video.mp4
Moviepy - Building video test\final_video_without_subtitles.mp4.
MoviePy - Writing audio in final_video_without_subtitlesTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video test\final_video_without_subtitles.mp4



Moviepy - Done !
Moviepy - video ready test\final_video_without_subtitles.mp4
Video file successfully written to test\final_video.mp4
Video file successfully written to test\final_video_without_subtitles.mp4
Video successfully saved to: test\final_video.mp4
Video composition completed successfully

Success! Video saved at: c:\Users\rajad\AptSmart\story\MM_StoryAgent\test\final_video.mp4
Could not copy video back to source directory: WindowsPath('test/final_video.mp4') and WindowsPath('test/final_video.mp4') are the same file
